## American Sobol

In [1]:
import numpy as np
from scipy.stats import norm
# Use numpy's polynomial tools for regression
from numpy.polynomial import polynomial as P
import time

def timed(func):
    """Decorator to time a function."""
    def wrapper(*args, **kwargs):
        start = time.time()
        result = func(*args, **kwargs)
        duration = time.time() - start
        # Modify return based on whether func returns multiple values
        if isinstance(result, tuple):
            return (*result, duration)
        else:
            return result, duration
    return wrapper

# --- Helper Function for GBM Path Generation (Still uses S0 internally) ---
def generate_gbm_paths(S0, T, r, sigma, n_paths, n_steps):
    """Generates Geometric Brownian Motion paths starting from S0."""
    dt = T / n_steps
    dW = np.random.standard_normal(size=(n_steps, n_paths)) * np.sqrt(dt)
    drift = (r - 0.5 * sigma**2) * dt
    S = np.zeros((n_steps + 1, n_paths))
    S[0] = S0
    for t in range(1, n_steps + 1):
        S[t] = S[t - 1] * np.exp(drift + sigma * dW[t - 1])
    return S

# --- LSMC American Put Option Pricer (Requires S0) ---
@timed
def american_put_option_lsmc(S0, K, T, r, sigma, n_paths, n_steps, degree=3):
    """Prices an American put option using LSMC."""
    # Input validation
    if n_paths <= 0 or not isinstance(n_paths, int): raise ValueError("n_paths must be > 0")
    if n_steps <= 0 or not isinstance(n_steps, int): raise ValueError("n_steps must be > 0")
    if T <= 0: raise ValueError("T must be > 0")
    if degree < 1: raise ValueError("degree must be >= 1")
    if S0 < 0 or K < 0: raise ValueError("S0 and K must be non-negative")
    if sigma < 0: raise ValueError("sigma cannot be negative")

    dt = T / n_steps
    discount_factor = np.exp(-r * dt)
    S = generate_gbm_paths(S0, T, r, sigma, n_paths, n_steps)
    V = np.maximum(K - S[n_steps], 0) # Put payoff at maturity

    for t in range(n_steps - 1, 0, -1):
        V = V * discount_factor
        itm_mask = S[t] < K # Put ITM condition
        if np.sum(itm_mask) > 0:
            X = S[t, itm_mask]
            Y = V[itm_mask]
            try:
                coeffs = P.polyfit(X, Y, degree)
                estimated_continuation_value = P.polyval(X, coeffs)
            except (np.linalg.LinAlgError, ValueError) as e:
                 print(f"Warning: Put regression failed at step {t}. Using fallback. Error: {e}")
                 estimated_continuation_value = Y # Fallback

            exercise_value = K - X # Put immediate exercise value
            exercise_now_mask = exercise_value > estimated_continuation_value
            full_exercise_mask = np.zeros(n_paths, dtype=bool)
            full_exercise_mask[itm_mask] = exercise_now_mask
            V[full_exercise_mask] = K - S[t, full_exercise_mask] # Update value if exercised

    option_price = np.mean(V * discount_factor)
    return option_price

# --- LSMC American Call Option Pricer (Requires S0) ---
@timed
def american_call_option_lsmc(S0, K, T, r, sigma, n_paths, n_steps, degree=3):
    """Prices an American call option using LSMC."""
    # Input validation (same as put)
    if n_paths <= 0 or not isinstance(n_paths, int): raise ValueError("n_paths must be > 0")
    if n_steps <= 0 or not isinstance(n_steps, int): raise ValueError("n_steps must be > 0")
    if T <= 0: raise ValueError("T must be > 0")
    if degree < 1: raise ValueError("degree must be >= 1")
    if S0 < 0 or K < 0: raise ValueError("S0 and K must be non-negative")
    if sigma < 0: raise ValueError("sigma cannot be negative")

    dt = T / n_steps
    discount_factor = np.exp(-r * dt)
    S = generate_gbm_paths(S0, T, r, sigma, n_paths, n_steps)
    V = np.maximum(S[n_steps] - K, 0) # Call payoff at maturity

    for t in range(n_steps - 1, 0, -1):
        V = V * discount_factor
        itm_mask = S[t] > K # Call ITM condition
        if np.sum(itm_mask) > 0:
            X = S[t, itm_mask]
            Y = V[itm_mask]
            try:
                coeffs = P.polyfit(X, Y, degree)
                estimated_continuation_value = P.polyval(X, coeffs)
            except (np.linalg.LinAlgError, ValueError) as e:
                print(f"Warning: Call regression failed at step {t}. Using fallback. Error: {e}")
                estimated_continuation_value = Y # Fallback

            exercise_value = X - K # Call immediate exercise value
            exercise_now_mask = exercise_value > estimated_continuation_value
            full_exercise_mask = np.zeros(n_paths, dtype=bool)
            full_exercise_mask[itm_mask] = exercise_now_mask
            V[full_exercise_mask] = S[t, full_exercise_mask] - K # Update value if exercised

    option_price = np.mean(V * discount_factor)
    return option_price

# --- Black-Scholes/Black-76 Formulas (for European Comparison) ---
def black_scholes_put(S0, K, T, r, sigma):
    """Analytical Black-Scholes formula for European put option."""
    if T <= 0: return np.maximum(K - S0, 0)
    if sigma <= 0: return np.maximum(K * np.exp(-r * T) - S0, 0)
    d1 = (np.log(S0 / K) + (r + 0.5 * sigma**2) * T) / (sigma * np.sqrt(T))
    d2 = d1 - sigma * np.sqrt(T)
    put_price = K * np.exp(-r * T) * norm.cdf(-d2) - S0 * norm.cdf(-d1)
    return put_price

def black_76_call(F, K, T, r, sigma):
    """Analytical Black-76 formula for European call option on a forward/future."""
    # Equivalent to BS Call when S0 = F*exp(-rT) and no dividends
    if T <= 0: return np.maximum(F - K, 0)
    if sigma <= 0: return np.maximum(F - K, 0) * np.exp(-r * T)
    d1 = (np.log(F / K) + (0.5 * sigma**2) * T) / (sigma * np.sqrt(T))
    d2 = d1 - sigma * np.sqrt(T)
    call_price = np.exp(-r * T) * (F * norm.cdf(d1) - K * norm.cdf(d2))
    return call_price

# === ⚙️ Option Parameters (Input F) ===
F = 100.0  # INPUT Forward Price
K = 100.0
T = 1.0
r = 0.05
sigma = 0.2

# === Calculate S0 from F ===
# Assuming no dividends or other carry costs/benefits
if T > 0:
    S0 = F * np.exp(-r * T)
else:
    S0 = F # At T=0, S0 = F

# === 🔢 Simulation Control ===
n_paths = 50000   # Number of paths (increase for better accuracy)
n_steps = 100     # Number of time steps
poly_degree = 5   # Degree of polynomial for regression

print("\n--- American Option Pricing (LSMC from F Input) ---")
print(f"Input Forward Price (F):        {F:.8f}")
print(f"Strike Price (K):               {K:.2f}")
print(f"Time to Maturity (T):           {T:.2f} years")
print(f"Risk-Free Rate (r):             {r:.2f}")
print(f"Volatility (sigma):             {sigma:.2f}")
print(f"Derived Spot Price (S0):        {S0:.8f}")
print("-" * 40)
print(f"Simulation Paths:               {n_paths}")
print(f"Time Steps:                     {n_steps}")
print(f"Polynomial Degree:              {poly_degree}")
print("-" * 40)

# === 🚀 LSMC American Put Calculation ===
(lsmc_put_price, duration_put) = american_put_option_lsmc(S0, K, T, r, sigma, n_paths, n_steps, degree=poly_degree)
print(f"LSMC American Put Estimate:     {lsmc_put_price:.8f} (Time: {duration_put:.4f}s)")

# === Optional: Compare with European Put (Black-Scholes using derived S0) ===
bs_put_price = black_scholes_put(S0, K, T, r, sigma)
print(f"Black-Scholes European Put:     {bs_put_price:.8f}")
early_exercise_premium_put = lsmc_put_price - bs_put_price
if abs(bs_put_price) > 1e-10:
    premium_pct_put = (early_exercise_premium_put / bs_put_price) * 100
    print(f"Est. Put Early Exercise Premium: {early_exercise_premium_put:.8f} ({premium_pct_put:.4f}%)")
else:
     print(f"Est. Put Early Exercise Premium: {early_exercise_premium_put:.8f}")
print("-" * 40)

# === 🚀 LSMC American Call Calculation ===
(lsmc_call_price, duration_call) = american_call_option_lsmc(S0, K, T, r, sigma, n_paths, n_steps, degree=poly_degree)
print(f"LSMC American Call Estimate:    {lsmc_call_price:.8f} (Time: {duration_call:.4f}s)")

# === Optional: Compare with European Call (Black-76 using F) ===
bs_call_price = black_76_call(F, K, T, r, sigma)
print(f"Black-76 European Call:         {bs_call_price:.8f}")
early_exercise_premium_call = lsmc_call_price - bs_call_price
# Note: For non-dividend stocks, this premium should theoretically be zero.
# Small positive values might occur due to MC noise or simulation artifacts.
# Large positive values could indicate an issue.
if abs(bs_call_price) > 1e-10:
    premium_pct_call = (early_exercise_premium_call / bs_call_price) * 100
    print(f"Est. Call Early Exercise Premium:{early_exercise_premium_call:.8f} ({premium_pct_call:.4f}%)")
else:
     print(f"Est. Call Early Exercise Premium:{early_exercise_premium_call:.8f}")
print("-" * 40)


--- American Option Pricing (LSMC from F Input) ---
Input Forward Price (F):        100.00000000
Strike Price (K):               100.00
Time to Maturity (T):           1.00 years
Risk-Free Rate (r):             0.05
Volatility (sigma):             0.20
Derived Spot Price (S0):        95.12294245
----------------------------------------
Simulation Paths:               50000
Time Steps:                     100
Polynomial Degree:              5
----------------------------------------
LSMC American Put Estimate:     8.38629518 (Time: 3.2481s)
Black-Scholes European Put:     7.57708215
Est. Put Early Exercise Premium: 0.80921304 (10.6797%)
----------------------------------------
LSMC American Call Estimate:    7.56074242 (Time: 2.3827s)
Black-76 European Call:         7.57708215
Est. Call Early Exercise Premium:-0.01633972 (-0.2156%)
----------------------------------------


In [2]:
import numpy as np
from scipy.stats import norm, qmc # Import qmc
# Use numpy's polynomial tools for regression
from numpy.polynomial import polynomial as P
import time

def timed(func):
    """Decorator to time a function."""
    def wrapper(*args, **kwargs):
        start = time.time()
        result = func(*args, **kwargs)
        duration = time.time() - start
        if isinstance(result, tuple):
            return (*result, duration)
        else:
            return result, duration
    return wrapper

# --- Helper Function for GBM Path Generation (Now with Sobol option) ---
def generate_gbm_paths(S0, T, r, sigma, n_paths, n_steps, use_sobol=False):
    """
    Generates Geometric Brownian Motion paths starting from S0.
    Can use standard PRNG or Sobol sequences.
    """
    dt = T / n_steps

    # Generate random increments
    if use_sobol:
        # --- Sobol Implementation ---
        if not (n_paths & (n_paths - 1) == 0) and n_paths > 0: # Check if power of 2
             print(f"Warning: n_paths ({n_paths}) is not a power of 2. Sobol performance might be suboptimal.")
        if n_steps <= 0:
            raise ValueError("n_steps must be positive for Sobol generation.")

        sampler = qmc.Sobol(d=n_steps, scramble=True) # Dimension is number of steps
        sobol_uniform = sampler.random(n=n_paths) # Shape: (n_paths, n_steps)

        # Clip values to avoid norm.ppf(0) or norm.ppf(1) -> inf/-inf
        # Small epsilon ensures values are within (eps, 1-eps)
        eps = 1e-15
        sobol_uniform = np.clip(sobol_uniform, eps, 1 - eps)

        sobol_normal = norm.ppf(sobol_uniform) # Convert uniform[0,1] to standard normal
        # Transpose to get shape (n_steps, n_paths) as expected below
        dW_sobol = sobol_normal.T
        dW = dW_sobol * np.sqrt(dt)
        # --- End Sobol ---
    else:
        # --- Standard PRNG Implementation ---
        dW_std = np.random.standard_normal(size=(n_steps, n_paths))
        dW = dW_std * np.sqrt(dt)
        # --- End PRNG ---

    # Precompute drift term
    drift = (r - 0.5 * sigma**2) * dt

    # Initialize paths
    S = np.zeros((n_steps + 1, n_paths))
    S[0] = S0 # Start simulation from S0

    # Simulate paths
    for t in range(1, n_steps + 1):
        # Ensure dW has the correct index for the loop (t-1 corresponds to the t'th step's increment)
        S[t] = S[t - 1] * np.exp(drift + sigma * dW[t - 1])

    return S # Shape: (n_steps + 1, n_paths)


# --- LSMC American Put Option Pricer (Requires S0, accepts use_sobol flag) ---
@timed
def american_put_option_lsmc(S0, K, T, r, sigma, n_paths, n_steps, degree=3, use_sobol=False):
    """Prices an American put option using LSMC."""
    # Input validation
    if n_paths <= 0 or not isinstance(n_paths, int): raise ValueError("n_paths must be > 0")
    if n_steps <= 0 or not isinstance(n_steps, int): raise ValueError("n_steps must be > 0")
    if T <= 0: raise ValueError("T must be > 0")
    if degree < 1: raise ValueError("degree must be >= 1")
    if S0 < 0 or K < 0: raise ValueError("S0 and K must be non-negative")
    if sigma < 0: raise ValueError("sigma cannot be negative")

    dt = T / n_steps
    discount_factor = np.exp(-r * dt)
    # Generate paths using the specified method (PRNG or Sobol)
    S = generate_gbm_paths(S0, T, r, sigma, n_paths, n_steps, use_sobol=use_sobol)
    V = np.maximum(K - S[n_steps], 0) # Put payoff at maturity

    for t in range(n_steps - 1, 0, -1):
        V = V * discount_factor
        itm_mask = S[t] < K # Put ITM condition
        # Ensure there are enough ITM points for regression (at least degree + 1)
        if np.sum(itm_mask) > degree:
            X = S[t, itm_mask]
            Y = V[itm_mask]
            try:
                coeffs = P.polyfit(X, Y, degree)
                estimated_continuation_value = P.polyval(X, coeffs)
            except (np.linalg.LinAlgError, ValueError) as e:
                 print(f"Warning: Put regression failed at step {t} with {np.sum(itm_mask)} ITM points. Using fallback. Error: {e}")
                 estimated_continuation_value = Y # Fallback

            exercise_value = K - X # Put immediate exercise value
            # Ensure comparison is valid even with fallback
            exercise_now_mask = exercise_value > estimated_continuation_value
            full_exercise_mask = np.zeros(n_paths, dtype=bool)
            full_exercise_mask[itm_mask] = exercise_now_mask
            V[full_exercise_mask] = K - S[t, full_exercise_mask] # Update value if exercised
        # else: # Handle cases with too few points for regression (optional)
        #    # e.g., assume holding is optimal if cannot regress
        #    pass # V remains the discounted value

    option_price = np.mean(V * discount_factor)
    return option_price

# --- LSMC American Call Option Pricer (Requires S0, accepts use_sobol flag) ---
@timed
def american_call_option_lsmc(S0, K, T, r, sigma, n_paths, n_steps, degree=3, use_sobol=False):
    """Prices an American call option using LSMC."""
    # Input validation
    if n_paths <= 0 or not isinstance(n_paths, int): raise ValueError("n_paths must be > 0")
    if n_steps <= 0 or not isinstance(n_steps, int): raise ValueError("n_steps must be > 0")
    if T <= 0: raise ValueError("T must be > 0")
    if degree < 1: raise ValueError("degree must be >= 1")
    if S0 < 0 or K < 0: raise ValueError("S0 and K must be non-negative")
    if sigma < 0: raise ValueError("sigma cannot be negative")

    dt = T / n_steps
    discount_factor = np.exp(-r * dt)
    # Generate paths using the specified method (PRNG or Sobol)
    S = generate_gbm_paths(S0, T, r, sigma, n_paths, n_steps, use_sobol=use_sobol)
    V = np.maximum(S[n_steps] - K, 0) # Call payoff at maturity

    for t in range(n_steps - 1, 0, -1):
        V = V * discount_factor
        itm_mask = S[t] > K # Call ITM condition
        # Ensure there are enough ITM points for regression (at least degree + 1)
        if np.sum(itm_mask) > degree:
            X = S[t, itm_mask]
            Y = V[itm_mask]
            try:
                coeffs = P.polyfit(X, Y, degree)
                estimated_continuation_value = P.polyval(X, coeffs)
            except (np.linalg.LinAlgError, ValueError) as e:
                print(f"Warning: Call regression failed at step {t} with {np.sum(itm_mask)} ITM points. Using fallback. Error: {e}")
                estimated_continuation_value = Y # Fallback

            exercise_value = X - K # Call immediate exercise value
            # Ensure comparison is valid even with fallback
            exercise_now_mask = exercise_value > estimated_continuation_value
            full_exercise_mask = np.zeros(n_paths, dtype=bool)
            full_exercise_mask[itm_mask] = exercise_now_mask
            V[full_exercise_mask] = S[t, full_exercise_mask] - K # Update value if exercised
        # else: # Handle cases with too few points for regression (optional)
        #    pass # V remains the discounted value

    option_price = np.mean(V * discount_factor)
    return option_price

# --- Black-Scholes/Black-76 Formulas (for European Comparison) ---
# (Keep these as they were)
def black_scholes_put(S0, K, T, r, sigma):
    if T <= 0: return np.maximum(K - S0, 0)
    if sigma <= 0: return np.maximum(K * np.exp(-r * T) - S0, 0)
    d1 = (np.log(S0 / K) + (r + 0.5 * sigma**2) * T) / (sigma * np.sqrt(T))
    d2 = d1 - sigma * np.sqrt(T)
    put_price = K * np.exp(-r * T) * norm.cdf(-d2) - S0 * norm.cdf(-d1)
    return put_price

def black_76_call(F, K, T, r, sigma):
    if T <= 0: return np.maximum(F - K, 0)
    if sigma <= 0: return np.maximum(F - K, 0) * np.exp(-r * T)
    d1 = (np.log(F / K) + (0.5 * sigma**2) * T) / (sigma * np.sqrt(T))
    d2 = d1 - sigma * np.sqrt(T)
    call_price = np.exp(-r * T) * (F * norm.cdf(d1) - K * norm.cdf(d2))
    return call_price

# === Option Parameters (Input F) ===
F = 100.0
K = 100.0
T = 1.0
r = 0.05
sigma = 0.2

# === Calculate S0 from F ===
if T > 0: S0 = F * np.exp(-r * T)
else: S0 = F

# === Simulation Control ===
# Use a power of 2 for n_paths when using Sobol
n_paths = 2**17   # e.g., 65536 paths (adjust as needed for performance/accuracy)
n_steps = 52     # Number of time steps
poly_degree = 5   # Degree of polynomial for regression
use_sobol = True # <<<--- Set to True to use Sobol sequences

print("\n--- American Option Pricing (LSMC from F Input) ---")
print(f"Input Forward Price (F):        {F:.8f}")
print(f"Strike Price (K):               {K:.2f}")
print(f"Time to Maturity (T):           {T:.2f} years")
print(f"Risk-Free Rate (r):             {r:.2f}")
print(f"Volatility (sigma):             {sigma:.2f}")
print(f"Derived Spot Price (S0):        {S0:.8f}")
print("-" * 40)
print(f"Simulation Method:              {'Sobol QMC' if use_sobol else 'Standard PRNG'}")
print(f"Simulation Paths (n_paths):     {n_paths}")
print(f"Time Steps (n_steps):           {n_steps}")
print(f"Polynomial Degree:              {poly_degree}")
print("-" * 40)

# === LSMC American Put Calculation ===
(lsmc_put_price, duration_put) = american_put_option_lsmc(
    S0, K, T, r, sigma, n_paths, n_steps, degree=poly_degree, use_sobol=use_sobol
)
print(f"LSMC American Put Estimate:     {lsmc_put_price:.8f} (Time: {duration_put:.4f}s)")

# === Optional: Compare with European Put ===
bs_put_price = black_scholes_put(S0, K, T, r, sigma)
print(f"Black-Scholes European Put:     {bs_put_price:.8f}")
early_exercise_premium_put = lsmc_put_price - bs_put_price
if abs(bs_put_price) > 1e-10:
    premium_pct_put = (early_exercise_premium_put / bs_put_price) * 100
    print(f"Est. Put Early Exercise Premium: {early_exercise_premium_put:.8f} ({premium_pct_put:.4f}%)")
else:
     print(f"Est. Put Early Exercise Premium: {early_exercise_premium_put:.8f}")
print("-" * 40)

# === LSMC American Call Calculation ===
(lsmc_call_price, duration_call) = american_call_option_lsmc(
    S0, K, T, r, sigma, n_paths, n_steps, degree=poly_degree, use_sobol=use_sobol
)
print(f"LSMC American Call Estimate:    {lsmc_call_price:.8f} (Time: {duration_call:.4f}s)")

# === Optional: Compare with European Call ===
bs_call_price = black_76_call(F, K, T, r, sigma)
print(f"Black-76 European Call:         {bs_call_price:.8f}")
early_exercise_premium_call = lsmc_call_price - bs_call_price
if abs(bs_call_price) > 1e-10:
    premium_pct_call = (early_exercise_premium_call / bs_call_price) * 100
    print(f"Est. Call Early Exercise Premium:{early_exercise_premium_call:.8f} ({premium_pct_call:.4f}%)")
else:
     print(f"Est. Call Early Exercise Premium:{early_exercise_premium_call:.8f}")
print("-" * 40)


--- American Option Pricing (LSMC from F Input) ---
Input Forward Price (F):        100.00000000
Strike Price (K):               100.00
Time to Maturity (T):           1.00 years
Risk-Free Rate (r):             0.05
Volatility (sigma):             0.20
Derived Spot Price (S0):        95.12294245
----------------------------------------
Simulation Method:              Sobol QMC
Simulation Paths (n_paths):     131072
Time Steps (n_steps):           52
Polynomial Degree:              5
----------------------------------------
LSMC American Put Estimate:     8.37242585 (Time: 6.5009s)
Black-Scholes European Put:     7.57708215
Est. Put Early Exercise Premium: 0.79534370 (10.4967%)
----------------------------------------
LSMC American Call Estimate:    7.56851428 (Time: 4.7981s)
Black-76 European Call:         7.57708215
Est. Call Early Exercise Premium:-0.00856786 (-0.1131%)
----------------------------------------
